In [1]:
from __future__ import print_function
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql.functions import col
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession, HiveContext ,Row, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, monotonically_increasing_id, current_date, date_format
from pyspark.sql.types import StructType, StructField, FloatType, StringType, ArrayType, IntegerType
from pyspark.sql import Row
from datetime import date
from dateutil.relativedelta import relativedelta
import datetime as dt
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import substring
from pyspark.sql.functions import countDistinct, collect_set

# Starting the session
spark = SparkSession\
    .builder\
    .appName("PythonSQL")\
    .config("spark.sql.parquet.datetimeRebaseModeInWrite","LEGACY")\
    .config("spark.sql.legacy.timeParserPolicy","LEGACY")\
    .config("spark.sql.execution.arrow.enabled", "true")\
    .getOrCreate()

Setting spark.hadoop.yarn.resourcemanager.principal to Shout5295


In [3]:
df  = spark.table("staging_zone.hpsv3_mw_cr_transaction_view").dropDuplicates()
df

DataFrame[SHADOW_ACCOUNT_NBR: string, CLIENT_CODE: string, CARD_NUMBER: string, TRANSACTION_DATE: timestamp, TRANSACTION_SIGN: string, BILLING_CURRENCY: string, BILLING_AMOUNT: decimal(18,3), MCC: string, TRANSACTION_CODE: string, TRANSACTION_DESCRIPTION: string, ACRONYM: string, REVERSAL_FLAG: string, REVERSAL_REASON_CODE: string, MICROFILM_REF_NUMBER: string, MICROFILM_REF_SEQ: decimal(4,0), SETTLEMENT_CURRENCY: string, SETTLEMENT_AMOUNT: decimal(18,3), CITY_CODE: string, NETWORK_CODE: string, PROCESSING_DATE: timestamp, hive_insertion_date: date, hive_insertion_time: timestamp, hive_insertion_year: int, hive_insertion_month: int, hive_insertion_day: int]

In [15]:
df.withColumn('month', F.month('transaction_date')).filter(col('SHADOW_ACCOUNT_NBR')=='048100718780').orderBy('transaction_date').show(vertical = True)

-RECORD 0---------------------------------------
 SHADOW_ACCOUNT_NBR      | 048100718780         
 CLIENT_CODE             | 27909292105274000    
 CARD_NUMBER             | 4150********2466     
 TRANSACTION_DATE        | 2025-03-12 16:51:35  
 TRANSACTION_SIGN        | D                    
 BILLING_CURRENCY        | 818                  
 BILLING_AMOUNT          | 500.000              
 MCC                     | 5999                 
 TRANSACTION_CODE        | 21                   
 TRANSACTION_DESCRIPTION | Subscript.  Fees     
 ACRONYM                 | null                 
 REVERSAL_FLAG           | N                    
 REVERSAL_REASON_CODE    | null                 
 MICROFILM_REF_NUMBER    | 71111115076101155... 
 MICROFILM_REF_SEQ       | 1                    
 SETTLEMENT_CURRENCY     | null                 
 SETTLEMENT_AMOUNT       | null                 
 CITY_CODE               | null                 
 NETWORK_CODE            | 01                   
 PROCESSING_DATE    

In [ ]:
df.filter(col('SHADOW_ACCOUNT_NBR')=='048100718780').orderBy('transaction_date').show(vertical = True)

-RECORD 0--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-03-12   
-RECORD 1--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-01   
-RECORD 2--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-02   
-RECORD 3--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-02   
-RECORD 4--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-02   
-RECORD 5--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-02   
-RECORD 6--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-02   
-RECORD 7--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-04-30   
-RECORD 8--------------------------
 SHADOW_ACCOUNT_NBR | 048100718780 
 transaction_date   | 2025-05-28   
-RECORD 9-------------------

In [4]:
df.count()

42439248

In [6]:
df.select(F.countDistinct('SHADOW_ACCOUNT_NBR')).show()

+----------------------------------+
|count(DISTINCT SHADOW_ACCOUNT_NBR)|
+----------------------------------+
|                           1178868|
+----------------------------------+



In [2]:
df  = spark.table("analytics_zone_retail_cross_selling.final_accounts_v22")
df

Hive Session ID = d525dacb-13d9-4a0b-8764-f05a682cf50e


DataFrame[SHADOW_ACCOUNT_NBR: string, Year: int, Month: int, purchases_trx_count: bigint, total_amount: decimal(28,3), total_amount_purchase: decimal(28,3), min_trx_date: timestamp, max_trx_date: timestamp, trx_count: bigint, prev_max_trx_date: timestamp, delay_days: int, first_non_null_trx_date: timestamp, Time_since_first_transaction: int, Count_Transactions_back: int, avg_delay_gaps: double, stddev_gaps: double, client_code: string, statement_date: timestamp, credit_limit: decimal(18,3), opening_balance: decimal(18,3), closing_balance: decimal(18,3), Remaining_closing_Balance: decimal(19,3), CURRENT_BALANCE_PURCHASE: decimal(18,3), CURRENT_BALANCE_CASH: decimal(18,3), CURRENT_BALANCE_TRANSFER: decimal(18,3), CURRENT_BALANCE_ECOM: decimal(18,3), CURRENT_BALANCE_CHEQUE: decimal(18,3), CURRENT_BALANCE_INTEREST: decimal(18,3), CURRENT_BALANCE_INSTALLMENT: decimal(18,3), CURRENT_BALANCE_FEES: decimal(18,3), CURRENT_BALANCE_TAX: decimal(18,3), CURRENT_BALANCE_REFUND: decimal(18,3), CURREN

In [5]:
df.printSchema()

root
 |-- SHADOW_ACCOUNT_NBR: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- purchases_trx_count: long (nullable = true)
 |-- total_amount: decimal(28,3) (nullable = true)
 |-- total_amount_purchase: decimal(28,3) (nullable = true)
 |-- min_trx_date: timestamp (nullable = true)
 |-- max_trx_date: timestamp (nullable = true)
 |-- trx_count: long (nullable = true)
 |-- prev_max_trx_date: timestamp (nullable = true)
 |-- delay_days: integer (nullable = true)
 |-- first_non_null_trx_date: timestamp (nullable = true)
 |-- Time_since_first_transaction: integer (nullable = true)
 |-- Count_Transactions_back: integer (nullable = true)
 |-- avg_delay_gaps: double (nullable = true)
 |-- stddev_gaps: double (nullable = true)
 |-- client_code: string (nullable = true)
 |-- statement_date: timestamp (nullable = true)
 |-- credit_limit: decimal(18,3) (nullable = true)
 |-- opening_balance: decimal(18,3) (nullable = true)
 |-- closing_balance:

In [3]:
df.select('balance_and_inactivity_P_counter').distinct().show()

+--------------------------------+
|balance_and_inactivity_P_counter|
+--------------------------------+
|                              31|
|                              34|
|                              28|
|                              26|
|                              27|
|                              12|
|                              22|
|                               1|
|                              13|
|                               6|
|                              16|
|                               3|
|                              20|
|                               5|
|                              19|
|                              15|
|                               9|
|                              17|
|                              35|
|                               4|
+--------------------------------+
only showing top 20 rows



In [4]:
df.filter(col('SHADOW_ACCOUNT_NBR')=='555152175658').orderBy('period_date').show(50,vertical=True)

-RECORD 0------------------------------------------------
 SHADOW_ACCOUNT_NBR               | 555152175658         
 Year                             | 2022                 
 Month                            | 6                    
 purchases_trx_count              | 0                    
 total_amount                     | null                 
 total_amount_purchase            | 0.000                
 min_trx_date                     | null                 
 max_trx_date                     | null                 
 trx_count                        | null                 
 prev_max_trx_date                | null                 
 delay_days                       | null                 
 first_non_null_trx_date          | null                 
 Time_since_first_transaction     | null                 
 Count_Transactions_back          | null                 
 avg_delay_gaps                   | null                 
 stddev_gaps                      | null                 
 client_code  

## Validation

In [2]:
df  = spark.table("staging_zone.hpsv3_transaction_hist")
 
transactions = (
    df.withColumn("transaction_date", F.to_date(F.col("DATE_CREATE"), "yyyy-MM-dd"))
)
transactions = transactions.filter(F.col("TRANSACTION_DATE") >= "2018-06-30")
transactions = transactions.withColumn("Year", F.year(transactions["transaction_date"])).withColumn(
        "Month", F.month(transactions["transaction_date"])
    )
transactions = transactions.select('CARD_NUMBER','MICROFILM_REF_NUMBER','Year',
                                   'Month', 'TRANSACTION_DATE', 'TRANSACTION_SIGN',
                                   'LOCAL_AMOUNT','EXPIRY_DATE', 'TRANSACTION_WORDING')


Hive Session ID = 07bcb900-ef3a-40d9-8467-a301347ae13b


In [23]:
df_link = spark.table("staging_zone.hpsv3_accounts_link")\
        .select(
            F.col("ENTITY_ID").alias("CARD_NUMBER"),
            F.col("ACCOUNT_NUMBER").alias("SHADOW_ACCOUNT_NBR"),
            F.col("OPENING_DATE")
        ).distinct()

In [20]:
df_link.filter(F.col("SHADOW_ACCOUNT_NBR") == "799178781293").show()

+----------------+------------------+-------------------+
|     CARD_NUMBER|SHADOW_ACCOUNT_NBR|       OPENING_DATE|
+----------------+------------------+-------------------+
|2244991404251102|      799178781293|2021-10-04 00:00:00|
+----------------+------------------+-------------------+



In [30]:
transactions.filter(F.col("CARD_NUMBER") == 2244991404251102).orderBy("TRANSACTION_DATE").show()

+----------------+--------------------+----+-----+----------------+----------------+------------+-------------------+-------------------+
|     CARD_NUMBER|MICROFILM_REF_NUMBER|Year|Month|TRANSACTION_DATE|TRANSACTION_SIGN|LOCAL_AMOUNT|        EXPIRY_DATE|TRANSACTION_WORDING|
+----------------+--------------------+----+-----+----------------+----------------+------------+-------------------+-------------------+
|2244991404251102|00000000000000000...|2021|   10|      2021-10-16|               D|    9948.000|2023-10-01 00:00:00|           Purchase|
|2244991404251102|00000000000000000...|2021|   10|      2021-10-16|               D|   27723.000|2023-10-01 00:00:00|           Purchase|
|2244991404251102|00000000000000000...|2021|   11|      2021-11-07|               D|    2300.000|2023-10-01 00:00:00|           Purchase|
|2244991404251102|00000000000000000...|2022|    1|      2022-01-23|               D|       1.000|               null|           Purchase|
|2244991404251102|0000000000000000

### tEST WITH NEW tRANSACTIONS TABLE

In [5]:
df_trxn =spark.table("staging_zone.hpsv3_mw_cr_transaction_view_vb")
df_trxn.filter(F.col("CARD_NUMBER") == 2244991404251102).select('SHADOW_ACCOUNT_NBR','CARD_NUMBER',
                                                                'MICROFILM_REF_NUMBER',
                                                                'TRANSACTION_DATE', 'TRANSACTION_SIGN',
                                                                'BILLING_AMOUNT', 'TRANSACTION_DESCRIPTION')\
                        .orderBy("TRANSACTION_DATE").show(truncate =False,n=500)
            

+------------------+----------------+-----------------------+-------------------+----------------+--------------+-----------------------+
|SHADOW_ACCOUNT_NBR|CARD_NUMBER     |MICROFILM_REF_NUMBER   |TRANSACTION_DATE   |TRANSACTION_SIGN|BILLING_AMOUNT|TRANSACTION_DESCRIPTION|
+------------------+----------------+-----------------------+-------------------+----------------+--------------+-----------------------+
|799178781293      |2244991404251102|00000000000000000005020|2021-10-31 00:00:00|D               |1384.050      |Instalment             |
|799178781293      |2244991404251102|00000000000000000006025|2021-10-31 00:00:00|D               |496.650       |Instalment             |
|799178781293      |2244991404251102|00000000000000000006422|2021-11-30 00:00:00|D               |114.830       |Instalment             |
|799178781293      |2244991404251102|00000000000000000006025|2021-11-30 00:00:00|D               |496.650       |Instalment             |
|799178781293      |22449914042511

In [ ]:
spark

# Test

In [2]:
df_sh = spark.table("analytics_zone_retail_cross_selling.ch1_accounts_prepared_view")
# df_sh.groupBy("period_date").count().orderBy("period_date", ascending = False).show(500)

Hive Session ID = 8b699ed6-87b3-4a97-a664-94fc4455ab2b


In [10]:
df_sh.filter(~F.isnull("closing_date"))\
    .select("SHADOW_ACCOUNT_NBR", "period_date","closing_date").orderBy("SHADOW_ACCOUNT_NBR","period_date", ascending = False).show(500)

+------------------+-----------+------------+
|SHADOW_ACCOUNT_NBR|period_date|closing_date|
+------------------+-----------+------------+
|      799178781293| 2025-12-01|  2024-12-26|
|      799178781293| 2025-11-01|  2024-12-26|
|      799178781293| 2025-10-01|  2024-12-26|
|      799178781293| 2025-09-01|  2024-12-26|
|      799178781293| 2025-08-01|  2024-12-26|
|      799178781293| 2025-07-01|  2024-12-26|
|      799178781293| 2025-06-01|  2024-12-26|
|      799178781293| 2025-05-01|  2024-12-26|
|      799178781293| 2025-04-01|  2024-12-26|
|      799178781293| 2025-03-01|  2024-12-26|
|      799178781293| 2025-02-01|  2024-12-26|
|      799178781293| 2025-01-01|  2024-12-26|
|      799178781293| 2024-12-01|  2024-12-26|
|      799178781293| 2024-11-01|  2024-12-26|
|      799178781293| 2024-10-01|  2024-12-26|
|      799178781293| 2024-09-01|  2024-12-26|
|      799178781293| 2024-08-01|  2024-12-26|
|      799178781293| 2024-07-01|  2024-12-26|
|      799178781293| 2024-06-01|  

In [3]:
df_sh.filter(F.col("SHADOW_ACCOUNT_NBR") == 799178781293 ).orderBy("period_date", ascending = True).show(500)

+------------------+----+-----+---------------------+-------------------+------------------+------------+---------------------+--------------------+---------+-----------------+-------------------+------------+---------------+---------------+-------------------------+------------------------+--------------------+------------------------+--------------------+----------------------+------------------------+---------------------------+--------------------+-------------------+----------------------+-----------------------+-----------------+-----------------+-------------------+---------------------+--------------------------+------------+-----------+
|SHADOW_ACCOUNT_NBR|Year|Month|last_transaction_date|purchases_trx_count|deposits_trx_count|total_amount|total_amount_purchase|total_amount_deposit|trx_count|      client_code|     statement_date|credit_limit|opening_balance|closing_balance|Remaining_closing_Balance|CURRENT_BALANCE_PURCHASE|CURRENT_BALANCE_CASH|CURRENT_BALANCE_TRANSFER|CURRENT_

# Main

In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

import warnings
warnings.filterwarnings("ignore")
sys.path.append("/home/cdsw")

from pipelines.Data_Preparation import DataPreparation
from Development.utils.helpers import *

def main():
    spark = create_spark()
    pipeline = DataPreparation(spark)
    pipeline.run()
    spark.stop()
    
if __name__ == "__main__":
    main()
    


Setting spark.hadoop.yarn.resourcemanager.principal to melameiry
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Hive Session ID = 9f562f7a-3506-4b58-8610-77c24e216d1b


 __Successfully loading


In [16]:
df = spark.table("analytics_zone_retail_cross_selling.final_accounts_v22")
df.count()

9212724

In [17]:
df.select(F.countDistinct('SHADOW_ACCOUNT_NBR')).show()

+----------------------------------+
|count(DISTINCT SHADOW_ACCOUNT_NBR)|
+----------------------------------+
|                            255909|
+----------------------------------+



In [14]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

df_sh = spark.table("staging_zone.hpsv3_shadow_account").filter(F.col("hive_insertion_date") == "2025-05-03")
df_sh = df_sh.select("SHADOW_ACCOUNT_NBR", "CANCELLATION_DATE", "AGREEMENT_DATE", "ADMINISTRATIVE_STATUS","ADMINISTRATIVE_STATUS_DATE")

df_sh = df_sh.withColumn(
    "closing_date",
    F.when(
        F.col("ADMINISTRATIVE_STATUS").isin([6, 7]),
        F.col("ADMINISTRATIVE_STATUS_DATE")
    ).otherwise(None).cast(DateType())
)


In [11]:
df_sh.groupBy("closing_date").count().orderBy("closing_date", ascending = False).show(500)

+------------+-----+
|closing_date|count|
+------------+-----+
|  2025-05-04|  190|
|  2025-05-03|   11|
|  2025-05-02|   10|
|  2025-05-01|    2|
|  2025-04-30|  971|
|  2025-04-29|  826|
|  2025-04-28|  874|
|  2025-04-27|  850|
|  2025-04-26|    4|
|  2025-04-23|  835|
|  2025-04-22|  405|
|  2025-04-21|    1|
|  2025-04-20|   36|
|  2025-04-19|  103|
|  2025-04-18|    1|
|  2025-04-17|  684|
|  2025-04-16| 1290|
|  2025-04-15|  784|
|  2025-04-14| 1569|
|  2025-04-13|  882|
|  2025-04-10|  730|
|  2025-04-09|  937|
|  2025-04-08|  608|
|  2025-04-07|  504|
|  2025-04-06|  359|
|  2025-04-04|    1|
|  2025-04-03|  160|
|  2025-04-02|  135|
|  2025-03-29|  138|
|  2025-03-27|  950|
|  2025-03-26|  618|
|  2025-03-25|  778|
|  2025-03-24|  473|
|  2025-03-23|  538|
|  2025-03-22|  149|
|  2025-03-21|   36|
|  2025-03-20|  563|
|  2025-03-19|  560|
|  2025-03-18|  824|
|  2025-03-17|  606|
|  2025-03-16|  640|
|  2025-03-15|   49|
|  2025-03-14|    3|
|  2025-03-13|  871|
|  2025-03-12

In [15]:
df_sh.orderBy("closing_date", ascending = False).show(50)

+------------------+-----------------+-------------------+---------------------+--------------------------+------------+
|SHADOW_ACCOUNT_NBR|CANCELLATION_DATE|     AGREEMENT_DATE|ADMINISTRATIVE_STATUS|ADMINISTRATIVE_STATUS_DATE|closing_date|
+------------------+-----------------+-------------------+---------------------+--------------------------+------------+
|      219024865429|             null|2016-02-02 00:00:00|                    6|       2025-05-04 00:00:00|  2025-05-04|
|      219026477921|             null|2016-05-10 00:00:00|                    6|       2025-05-04 00:00:00|  2025-05-04|
|      219026765720|             null|2016-05-29 00:00:00|                    6|       2025-05-04 00:00:00|  2025-05-04|
|      219027597829|             null|2016-07-21 00:00:00|                    6|       2025-05-04 00:00:00|  2025-05-04|
|      219033703624|             null|2017-05-30 00:00:00|                    6|       2025-05-04 00:00:00|  2025-05-04|
|      219038588522|            

In [16]:
df_sh.filter(F.isnull("closing_date")).count()

1218263

In [17]:
df_sh.filter(~F.isnull("closing_date")).count()

777075

### read prepared view until cross join 

In [19]:
df = spark.table("analytics_zone_retail_cross_selling.ch1_accounts_prepared_view")

In [21]:
df.printSchema()

root
 |-- SHADOW_ACCOUNT_NBR: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- last_transaction_date: timestamp (nullable = true)
 |-- purchases_trx_count: long (nullable = true)
 |-- deposits_trx_count: long (nullable = true)
 |-- total_amount: decimal(28,3) (nullable = true)
 |-- total_amount_purchase: decimal(28,3) (nullable = true)
 |-- total_amount_deposit: decimal(28,3) (nullable = true)
 |-- trx_count: long (nullable = true)
 |-- client_code: string (nullable = true)
 |-- statement_date: timestamp (nullable = true)
 |-- credit_limit: decimal(18,3) (nullable = true)
 |-- opening_balance: decimal(18,3) (nullable = true)
 |-- closing_balance: decimal(18,3) (nullable = true)
 |-- Remaining_closing_Balance: decimal(19,3) (nullable = true)
 |-- CURRENT_BALANCE_PURCHASE: decimal(18,3) (nullable = true)
 |-- CURRENT_BALANCE_CASH: decimal(18,3) (nullable = true)
 |-- CURRENT_BALANCE_TRANSFER: decimal(18,3) (nullable = true)
 |-- CUR

In [23]:
df.show()

11:44:25.074 [task-result-getter-0] ERROR org.apache.spark.scheduler.TaskSetManager - Task 0 in stage 10.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o175.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 10.0 failed 4 times, most recent failure: Lost task 0.3 in stage 10.0 (TID 28) (10.42.1.136 executor 1): java.io.FileNotFoundException: 
File does not exist: hdfs://cdpdevp/warehouse/tablespace/external/hive/analytics_zone_retail_cross_selling.db/ch1_accounts_prepared_view/part-00025-14b2afe4-ba7e-4923-83c7-94c70c14f29a-c000.snappy.parquet

It is possible the underlying files have been updated. You can explicitly invalidate
the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by
recreating the Dataset/DataFrame involved.
       
	at org.apache.spark.sql.errors.QueryExecutionErrors$.readCurrentFileNotFoundError(QueryExecutionErrors.scala:661)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:212)
	at org.apac

In [14]:
df_sh.filter(F.col('SHADOW_ACCOUNT_NBR')== '048100987700').select('SHADOW_ACCOUNT_NBR' , 'Year' , 'Month' , 'purchases_trx_count' ,'deposits_trx_count' ,'total_amount' , 'total_amount_purchase' ,'total_amount_deposit' , 'trx_count' , 'statement_date' ,'credit_limit').show()

11:28:30.903 [task-result-getter-0] ERROR org.apache.spark.scheduler.TaskSetManager - Task 0 in stage 6.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o166.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 6.0 failed 4 times, most recent failure: Lost task 0.3 in stage 6.0 (TID 12) (10.42.1.136 executor 1): java.io.FileNotFoundException: 
File does not exist: hdfs://cdpdevp/warehouse/tablespace/external/hive/analytics_zone_retail_cross_selling.db/ch1_accounts_prepared_view/part-00025-14b2afe4-ba7e-4923-83c7-94c70c14f29a-c000.snappy.parquet

It is possible the underlying files have been updated. You can explicitly invalidate
the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by
recreating the Dataset/DataFrame involved.
       
	at org.apache.spark.sql.errors.QueryExecutionErrors$.readCurrentFileNotFoundError(QueryExecutionErrors.scala:661)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.org$apache$spark$sql$execution$datasources$FileScanRDD$$anon$$readCurrentFile(FileScanRDD.scala:212)
	at org.apache

In [8]:
df_sh.filter(F.col('SHADOW_ACCOUNT_NBR')== '048100987700').select('SHADOW_ACCOUNT_NBR' , 'Year' , 'Month' , 'purchases_trx_count' ,'deposits_trx_count' ,'total_amount' , 'total_amount_purchase' ,'total_amount_deposit' , 'trx_count' , 'statement_date' ,'credit_limit').show()

+------------------+----+-----+-------------------+------------------+------------+---------------------+--------------------+---------+-------------------+------------+
|SHADOW_ACCOUNT_NBR|Year|Month|purchases_trx_count|deposits_trx_count|total_amount|total_amount_purchase|total_amount_deposit|trx_count|     statement_date|credit_limit|
+------------------+----+-----+-------------------+------------------+------------+---------------------+--------------------+---------+-------------------+------------+
|      048100987700|2024|    6|                  0|                 0|       0.000|                0.000|               0.000|        0|               null|        null|
|      048100987700|2023|   12|                  0|                 0|       0.000|                0.000|               0.000|        0|               null|        null|
|      048100987700|2024|    1|                  0|                 0|       0.000|                0.000|               0.000|        0|              

In [4]:
df = spark.range(1)

df = df.select(F.month('2025-03-31 22:00:00').alias('month'))
df.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `2025-03-31 22:00:00` cannot be resolved. Did you mean one of the following? [`id`].;
'Project [month('2025-03-31 22:00:00) AS month#11]
+- Range (0, 1, step=1, splits=Some(2))
